# codesearch — реранкер: ретривер vs ретривер+реранкер

Один прогон: корпус **10k** функций на ретривер (Qwen3-0.6B) → топ-**100** кандидатов → реранкер
(Qwen3-Reranker-0.6B, **reranker_max_length=2048**). Смотрим **средние метрики** по запросам и
сравниваем с чистым ретривером.

Ключевой вопрос: даёт ли реранкер прирост на первых местах (R@1, MRR, NDCG@10), если его НЕ душить
512 токенами. R@50/@100 почти не меняются — реранкер лишь пересортировывает пул, не добавляя в него.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**, затем Runtime → Run all.

In [ ]:
!nvidia-smi

In [ ]:
!rm -rf code-searching-engine
!git clone -b test/reranker-vanya https://github.com/PopovDanil/code-searching-engine.git
%cd code-searching-engine
!pip install -q -U sentence-transformers
!pip install -q rank-bm25 faiss-cpu tree-sitter tree-sitter-python tree-sitter-java tree-sitter-javascript tree-sitter-go tree-sitter-ruby tree-sitter-php datasets typer pyyaml

In [ ]:
import os
import re
import subprocess

# Языки и число запросов на язык. Реранкер медленный (100 кандидатов на запрос),
# поэтому по умолчанию python — самый показательный (там ретривер слабее всего на R@1).
# Добавь языков при желании: "python,java,go".
LANGUAGES = "python"
MAX_DATASET_RECORDS = "10000"   # корпус на ретривер
MAX_QUERIES = "100"             # запросов на язык (mean по ним)

env = os.environ.copy()
env["PYTHONPATH"] = "src"


def parse_results(text):
    res, current = {}, None
    tail = text.split("Evaluation Results:")[-1]
    for line in tail.splitlines():
        m_lang = re.match(r"^\s{2}(\S+):\s*$", line)
        m_val = re.match(r"^\s{4}(\S+):\s+([0-9.]+)\s*$", line)
        if m_lang:
            current = m_lang.group(1); res[current] = {}
        elif m_val and current is not None:
            res[current][m_val.group(1)] = float(m_val.group(2))
    return res


def run(config_path, label):
    print(f"\n{'='*60}\n  {label}  ({config_path})\n{'='*60}", flush=True)
    proc = subprocess.run(
        ["python", "-m", "cli", "evaluate",
         "--languages", LANGUAGES,
         "--max-dataset-records", MAX_DATASET_RECORDS,
         "--max-queries", MAX_QUERIES,
         "--config", config_path],
        env=env, capture_output=True, text=True,
    )
    out = proc.stdout + "\n" + proc.stderr
    parsed = parse_results(out)
    if proc.returncode == 0 and parsed:
        return parsed.get("overall", {})
    print("FAILED, хвост:\n", out[-3000:])
    return {}


emb = run("configs/vanya-embedding-only.yaml", "РЕТРИВЕР-ONLY")
rer = run("configs/vanya-reranker.yaml", "РЕТРИВЕР + РЕРАНКЕР (2048)")

In [ ]:
# Сравнение средних метрик: ретривер-only vs +реранкер, и дельта
METRICS = ["Recall@1", "Recall@5", "Recall@10", "Recall@50", "Recall@100", "MRR", "NDCG@10", "NDCG@100"]

print(f"{'Метрика':<12} {'ретривер':>10} {'+реранкер':>10} {'дельта':>10}")
print("-" * 46)
for m in METRICS:
    a = emb.get(m, float('nan')); b = rer.get(m, float('nan'))
    print(f"{m:<12} {a:>10.4f} {b:>10.4f} {b - a:>+10.4f}")

print("\nMarkdown для чата:\n")
print("| Метрика | ретривер | +реранкер | дельта |")
print("|---|---|---|---|")
for m in METRICS:
    a = emb.get(m, float('nan')); b = rer.get(m, float('nan'))
    print(f"| {m} | {a:.4f} | {b:.4f} | {b - a:+.4f} |")

## Как читать

- **R@1 / MRR / NDCG@10** должны заметно вырасти с реранкером — это его работа (поднять правильный
  ответ выше). Если прирост большой — 2048 лечит то, что душило 512.
- **R@100** практически не изменится — реранкер лишь пересортировывает те же 100 кандидатов.
- **R@50/R@10** могут подрасти, если реранкер вытаскивает ответ из глубины пула ближе к верху.